# Exp7.3.6 — Affine mean-head bridge

Analysis-only notebook. It reads finalized Exp7.3.6 artifacts and summarizes method-level effects of bias, scale conditioning, and LIF realization.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=None):
    p = Path.cwd() if start is None else Path(start)
    for candidate in [p, *p.parents]:
        if (candidate / 'AGENTS.md').exists():
            return candidate
    raise FileNotFoundError('Repository root not found')

REPO_ROOT = find_repo_root()
ROOT = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_7_3_6_affine_mean_head_bridge' / 'affine_mean_head_bridge_v1'
runs = pd.read_csv(ROOT / 'method_runs.csv')
summary = pd.read_csv(ROOT / 'method_summary.csv')
contrasts = pd.read_csv(ROOT / 'contrast_summary.csv')
bias = pd.read_csv(ROOT / 'bias_ablation.csv')
lif = pd.read_csv(ROOT / 'lif_realization_summary.csv')
history = pd.read_csv(ROOT / 'history_summary.csv')
source_checks = json.loads((ROOT / 'source_reproduction_checks.json').read_text())
manifest = json.loads((ROOT / 'manifest.json').read_text())


## Primary analog comparison

In [ ]:
display(summary[['method', 'analog_test_ba_mean', 'analog_test_ba_std', 'lif_test_ba_mean', 'lif_test_ba_std']])
display(contrasts)
print('P3 mean-affine reference:', source_checks.get('p3_mean_affine_test_ba'))
print('P7 whole-count affine reference:', source_checks.get('p7_wholecount_affine_test_ba'))


In [ ]:
plot_df = summary.set_index('method').loc[[
    'H0_raw_no_bias',
    'H1_raw_bias',
    'H2_scale_no_bias',
    'H3_scale_bias',
]]
ax = plot_df['analog_test_ba_mean'].plot(kind='bar', yerr=plot_df['analog_test_ba_std'], capsize=4)
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Exp7.3.6 analog mean-head comparison')
plt.tight_layout()
plt.show()


## Bias ablation

In [ ]:
display(bias[['method', 'seed', 'analog_test_ba', 'bias_off_test_ba', 'direct_bias_effect_pp']])
display(bias.groupby('method')['direct_bias_effect_pp'].agg(['mean', 'std']))


## Analog-to-spiking realization

In [ ]:
display(lif)
realization = lif.groupby('method')[['analog_test_ba', 'lif_test_ba', 'if_count_test_ba', 'if_charge_test_ba']].mean()
ax = realization.plot(kind='bar')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Analog vs IF/LIF realization')
plt.tight_layout()
plt.show()
print('Max beta=1 charge identity error:', manifest['max_if_charge_identity_abs_error'])


## Training histories

In [ ]:
for method, df in history.groupby('method'):
    plt.plot(df['epoch'], df['val_ba_mean'], label=method)
plt.xlabel('Epoch')
plt.ylabel('Validation balanced accuracy')
plt.title('Validation BA by head formulation')
plt.legend()
plt.tight_layout()
plt.show()
